# ScreamingFace quickstart

Six steps: inspect the public Leaderboards, connect a provider, run a Benchmark, read the
Report, publish its Candidate Result, and replay its URL4. The wider interface is covered in
`01_client_tour.ipynb`.

## Before running

From a terminal:

```bash
screamingface prepare draco  # first run only: download pinned Benchmark assets
screamingface up             # start Gateway :9105, Scoreboard :9106, and Engine :9108
screamingface status
```

Use `screamingface logs` to inspect startup failures and `screamingface down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [ ]:
import screamingface as sf

sf.configure(
    engine_url="http://127.0.0.1:9108",
    scoreboard_url="http://127.0.0.1:9106",
)

BENCHMARK_ID = "draco"

## 1 · Leaderboards

Leaderboard discovery reads from the independently seeded Scoreboard and does not require a
provider connection. Its registered boards may differ from the Engine's Benchmark catalogue.
Both values render as interactive, brand-system notebook widgets.

In [ ]:
leaderboards = sf.leaderboards.list()
leaderboards

In [ ]:
leaderboard = sf.leaderboards.get(BENCHMARK_ID, top=10)
leaderboard

## 2 · Connect

`sf.connect()` renders the Engine-backed provider panel. A key entered here goes to the SF
Engine for AI Gateway validation and encrypted storage; the notebook never retains it. On a
hosted Engine the panel asks for Cloudflare Access login first.

In [ ]:
sf.connect()

## 3 · Evaluate

`limit=1` selects one Case from canonical DRACO. The Benchmark still applies every rubric
criterion and all five canonical Judge passes, so this is an authentic one-Case rehearsal—not a
weakened smoke protocol. It is **not** comparable with a complete 100-Case DRACO result. Grading
can still make many paid calls; run it deliberately. While it runs, the live panel shows
progress,
model calls, tokens and cost.

In [ ]:
candidate = sf.Model("openrouter/google/gemini-3-flash-preview")

report = sf.evaluate(candidate, benchmark=BENCHMARK_ID, limit=1)

## 4 · Report

The Report renders score, pass rate, coverage, cost and tokens, with every Case and the
Judge's per-criterion reasoning underneath. **&darr; report.json** downloads the portable
artifact — the same complete JSON document `report.export()` writes to the notebook's working
directory.

In [ ]:
report

In [ ]:
artifact_path = report.export()
artifact_path

## 5 · Publish and retrieve

Publication accepts the evaluated `CandidateResult` directly. It derives the Benchmark id,
compiled URL4, models, the Benchmark-native score, timestamps, and idempotency key from that
immutable result — the score is submitted exactly as the Engine reported it, and the
Scoreboard ranks it without recalculating. Publication is independently opt-in so
**Run All** never changes the Scoreboard.

The local Scoreboard accepts writes without login. Hosted deployments may require an
edge-verified identity or keep score submission closed.

In [ ]:
PUBLISH_RESULT = False

submission = sf.leaderboards.submit(report.candidates.only) if PUBLISH_RESULT else None
submission if submission is not None else ("Set PUBLISH_RESULT = True to publish this result.")

In [ ]:
published_score = sf.leaderboards.get_score(submission.id) if submission is not None else None
published_score

In [ ]:
updated_leaderboard = (
    sf.leaderboards.get(BENCHMARK_ID, top=10) if submission is not None else leaderboard
)
updated_leaderboard

## 6 · Fork or replay the submitted URL4

`published_score.url4` is the raw evaluation expression stored by the Scoreboard. Its
`.to_python()` method returns an editable Model/Fusion and evaluation cell without spending.
Passing the URL4 itself to `sf.evaluate(...)` instead executes that exact, already
Benchmark-linked expression and returns a normal `Report`; do not pass `benchmark=` or `limit=`
again.

Replay is a fresh paid Evaluation and model output may differ, so it has its own opt-in guard.


In [ ]:
fork_python = published_score.url4.to_python() if published_score is not None else None
print(fork_python if fork_python is not None else "Publish a score to generate a fork.")

In [ ]:
REPLAY_RESULT = False

replayed_report = (
    sf.evaluate(published_score.url4) if REPLAY_RESULT and published_score is not None else None
)
replayed_report if replayed_report is not None else (
    "Set REPLAY_RESULT = True after publishing to run the stored URL4 again."
)